# 中证800 V68 模型更新频率稳定性实验

目标：验证“年度 champion + 半年度 challenger”是否比机械半年换新更稳。

这个 notebook 只回答模型更新问题：

- 年度 champion：训练截止到上一年 12 月，预测下一整年。
- 半年度 challenger：训练截止到当年 6 月，只和 champion 比较当年 7-12 月共同区间。
- 不导出 pkl，不做 JoinQuant 撮合回测，不改生产策略。

核心问题：如果 `M_2025-12` 对 2026 全年不错，`M_2026-06` 加了半年数据后，对 2026H2 是否稳定优于/接近旧模型？

In [ ]:
import os
import gc
import json
import pickle
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))

## 1. 配置

默认用当前主线口径：full 特征、LGB fixed120、top8_board_cap。  
`TRAIN_POLICIES` 默认同时测 expanding 与 rolling60m，方便判断是更新频率问题还是训练窗口问题。

In [ ]:
PROJECT_DIR = Path("/Users/youzou/Documents/New project/quant-research/机器学习策略")
OUT_DIR = PROJECT_DIR / "csi800_ml_v68_update_frequency_stability_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])

# Keep this aligned with V61/V65 unless explicitly testing strict boundary.
LABEL_BOUNDARY_MODE = "legacy_rebalance"  # legacy_rebalance / label_end_safe

TRAIN_START = "2019-01-01"
MIN_TRAIN_MONTHS = 36
TRAIN_POLICIES = ["expanding", "rolling60m"]
ROLLING_MONTHS_BY_POLICY = {"rolling60m": 60, "rolling72m": 72}

# Pseudo-real update years. A pair y means: champion cutoff=y-12, challenger cutoff=(y+1)-06, compare (y+1)-07..(y+1)-12.
FIRST_CHAMPION_YEAR = 2021
LAST_CHAMPION_YEAR = 2025

RANDOM_SIM_N = 500
RANDOM_SEED = 42

SLIPPAGE_RATE = 0.00246
OPEN_COMMISSION = 0.0003
CLOSE_COMMISSION = 0.0003
CLOSE_TAX = 0.001
BUY_COST_RATE = SLIPPAGE_RATE + OPEN_COMMISSION
SELL_COST_RATE = SLIPPAGE_RATE + CLOSE_COMMISSION + CLOSE_TAX

print("OUT_DIR:", OUT_DIR)
print("TRAIN_POLICIES:", TRAIN_POLICIES)
print("LABEL_BOUNDARY_MODE:", LABEL_BOUNDARY_MODE)
print("portfolio:", PORTFOLIO_RULE, STOCK_NUM, BOARD_CAPS_TEXT)

## 2. 特征与模型参数

In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

feature_manifest_df = pd.DataFrame([{
    "feature_variant": "full",
    "candidate_feature_count": len(FULL_FEATURE_COLS),
    "candidate_features": ",".join(FULL_FEATURE_COLS),
}])
feature_manifest_df.to_csv(OUT_DIR / "v68_feature_manifest.csv", index=False)
display_df(feature_manifest_df)

## 3. Helper 函数

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_max_drawdown(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "worst_month": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_max_drawdown(s),
        "worst_month": float(s.min()),
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def get_stock_board(stock):
    code = str(stock).split(".")[0]
    if code.startswith(("300", "301")):
        return "chinext"
    if code.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    board = get_stock_board(stock)
    if board not in board_caps:
        return True
    current = sum(1 for s in selected if get_stock_board(s) == board)
    return current < int(board_caps[board])


def build_board_capped_targets(sorted_stocks, target_num=STOCK_NUM, board_caps=BOARD_CAPS):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
            if len(selected) >= target_num:
                return selected
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= target_num:
                break
    return selected


def feature_set_jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / float(len(sa | sb))


def list_jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / float(len(sa | sb))

## 4. 数据加载与更新计划

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


def train_start_for_policy(policy, train_end):
    train_end = pd.Timestamp(train_end)
    if policy == "expanding":
        return pd.Timestamp(TRAIN_START)
    if policy in ROLLING_MONTHS_BY_POLICY:
        months = int(ROLLING_MONTHS_BY_POLICY[policy])
        return (train_end - pd.DateOffset(months=months - 1)).replace(day=1)
    raise ValueError("unknown train policy: " + str(policy))


def make_train_df(df_all, policy, train_end):
    train_end = pd.Timestamp(train_end)
    train_start = train_start_for_policy(policy, train_end)
    mask = (df_all[DATE_COL] >= train_start) & (df_all[DATE_COL] <= train_end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= train_end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy(), train_start


def build_update_plan(df_all):
    available_months = sorted(pd.to_datetime(df_all[DATE_COL].dropna().unique()))
    available_set = set(available_months)
    rows = []
    for policy in TRAIN_POLICIES:
        for champion_year in range(FIRST_CHAMPION_YEAR, LAST_CHAMPION_YEAR + 1):
            test_year = champion_year + 1
            champion_end = pd.Timestamp("%s-12-31" % champion_year)
            challenger_end = pd.Timestamp("%s-06-30" % test_year)
            h1_start = pd.Timestamp("%s-01-01" % test_year)
            h1_end = pd.Timestamp("%s-06-30" % test_year)
            h2_start = pd.Timestamp("%s-07-01" % test_year)
            h2_end = pd.Timestamp("%s-12-31" % test_year)
            h1_months = [m for m in available_months if m >= h1_start and m <= h1_end]
            h2_months = [m for m in available_months if m >= h2_start and m <= h2_end]
            champ_train_start = train_start_for_policy(policy, champion_end)
            chall_train_start = train_start_for_policy(policy, challenger_end)
            rows.append({
                "policy": policy,
                "champion_year": champion_year,
                "test_year": test_year,
                "champion_train_start": champ_train_start,
                "champion_train_end": champion_end,
                "challenger_train_start": chall_train_start,
                "challenger_train_end": challenger_end,
                "h1_start": h1_start,
                "h1_end": h1_end,
                "h2_start": h2_start,
                "h2_end": h2_end,
                "available_h1_months": len(h1_months),
                "available_h2_months": len(h2_months),
                "status": "ok" if len(h2_months) > 0 else "no_h2_data_yet",
            })
    return pd.DataFrame(rows)


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
update_plan_df = build_update_plan(df_all)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("next_date:", df_all["next_date"].min(), "->", df_all["next_date"].max())
update_plan_df.to_csv(OUT_DIR / "v68_update_plan.csv", index=False)
display_df(update_plan_df, 20)

## 5. 训练所有需要的 cutoff 模型

同一个 cutoff 在多个比较里会复用。

In [ ]:
def train_model_for_cutoff(df_all, policy, train_end):
    train_df, train_start = make_train_df(df_all, policy, train_end)
    if train_df.empty:
        raise ValueError("empty train_df policy=%s train_end=%s" % (policy, train_end))
    if train_df[DATE_COL].nunique() < MIN_TRAIN_MONTHS:
        raise ValueError("too few train months policy=%s train_end=%s months=%s" % (policy, train_end, train_df[DATE_COL].nunique()))
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, FULL_FEATURE_COLS)
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    model_id = "%s_cutoff%s" % (policy, pd.Timestamp(train_end).strftime("%Y%m%d"))
    return {
        "model_id": model_id,
        "policy": policy,
        "train_start": train_start,
        "train_end": pd.Timestamp(train_end),
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "feature_cols": feature_cols,
        "removed_cols": removed_cols,
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "model": trained["model"],
        "fill_values": trained["fill_values"],
    }

needed = []
for _, r in update_plan_df.iterrows():
    needed.append((r["policy"], pd.Timestamp(r["champion_train_end"])))
    needed.append((r["policy"], pd.Timestamp(r["challenger_train_end"])))
needed = sorted(set(needed), key=lambda x: (x[0], x[1]))

model_bank = {}
model_meta_rows = []
for policy, train_end in progress_iter(needed, total=len(needed), desc="train cutoff models"):
    try:
        m = train_model_for_cutoff(df_all, policy, train_end)
        model_bank[(policy, pd.Timestamp(train_end))] = m
        meta_row = {k: v for k, v in m.items() if k not in ["model", "fill_values", "feature_cols", "removed_cols"]}
        meta_row.update({
            "feature_cols": ",".join(m["feature_cols"]),
            "removed_features": ",".join(m["removed_cols"]),
        })
        model_meta_rows.append(meta_row)
        print("trained", m["model_id"], "months", m["train_months"], "features", m["feature_count"], "diag", m["diag_rank_ic"])
    except Exception as err:
        model_meta_rows.append({"policy": policy, "train_end": train_end, "error": str(err)})
        print("train failed", policy, train_end, err)

model_meta_df = pd.DataFrame(model_meta_rows)
model_meta_df.to_csv(OUT_DIR / "v68_model_meta.csv", index=False)
display_df(model_meta_df, 20)

## 6. 打分、构造 top8_board_cap、比较 frozen vs updated

In [ ]:
def score_panel_for_model(df_test, model_info, score_col):
    out = df_test[[STOCK_COL, DATE_COL, "next_date", TARGET_COL, INDUSTRY_COL]].copy()
    out[score_col] = score_with_model(df_test, model_info["model"], model_info["feature_cols"], model_info["fill_values"])
    return out


def random_percentile(month_df, selected, n_sim=RANDOM_SIM_N, seed=RANDOM_SEED):
    if len(month_df) == 0 or len(selected) == 0:
        return np.nan
    rng = np.random.RandomState(seed)
    stocks = month_df[STOCK_COL].astype(str).tolist()
    ret_map = dict(zip(month_df[STOCK_COL].astype(str), pd.to_numeric(month_df[TARGET_COL], errors="coerce")))
    selected_ret = np.nanmean([ret_map.get(s, np.nan) for s in selected])
    if pd.isnull(selected_ret):
        return np.nan
    vals = []
    arr = np.arange(len(stocks))
    for _ in range(int(n_sim)):
        perm = rng.permutation(arr)
        ordered = [stocks[i] for i in perm]
        picked = build_board_capped_targets(ordered, STOCK_NUM, BOARD_CAPS)
        if len(picked) == 0:
            continue
        vals.append(np.nanmean([ret_map.get(s, np.nan) for s in picked]))
    if len(vals) == 0:
        return np.nan
    return float((np.asarray(vals) <= selected_ret).mean())


def select_month_targets(gdf, score_col):
    m = gdf.dropna(subset=[score_col, TARGET_COL]).copy()
    if m.empty:
        return [], m
    sorted_stocks = m.sort_values(score_col, ascending=False)[STOCK_COL].astype(str).tolist()
    targets = build_board_capped_targets(sorted_stocks, STOCK_NUM, BOARD_CAPS)
    return targets, m


def calc_month_result(gdf, score_col, prefix, prev_targets=None):
    targets, m = select_month_targets(gdf, score_col)
    ret_map = dict(zip(m[STOCK_COL].astype(str), pd.to_numeric(m[TARGET_COL], errors="coerce")))
    gross_alpha = np.nanmean([ret_map.get(s, np.nan) for s in targets]) if targets else np.nan
    if prev_targets is None:
        overlap_prev = 0
    else:
        overlap_prev = len(set(targets) & set(prev_targets))
    target_count = len(targets)
    buy_turnover = 1.0 - overlap_prev / float(max(1, target_count))
    sell_turnover = buy_turnover
    trade_cost = buy_turnover * BUY_COST_RATE + sell_turnover * SELL_COST_RATE
    net_alpha = gross_alpha - trade_cost if not pd.isnull(gross_alpha) else np.nan
    actual_top20 = set(m.sort_values(TARGET_COL, ascending=False).head(20)[STOCK_COL].astype(str).tolist()) if not m.empty else set()
    hit_top20 = len(set(targets) & actual_top20) / float(max(1, target_count)) if targets else np.nan
    board_counts = {"main": 0, "chinext": 0, "star": 0}
    for s in targets:
        b = get_stock_board(s)
        board_counts[b] = board_counts.get(b, 0) + 1
    return {
        "%s_targets" % prefix: ",".join(targets),
        "%s_target_count" % prefix: target_count,
        "%s_gross_alpha_ret" % prefix: gross_alpha,
        "%s_net_alpha_ret" % prefix: net_alpha,
        "%s_trade_cost" % prefix: trade_cost,
        "%s_hit_top20" % prefix: hit_top20,
        "%s_random_alpha_percentile" % prefix: random_percentile(m, targets),
        "%s_rank_ic" % prefix: safe_rank_ic(m[score_col], m[TARGET_COL]) if len(m) else np.nan,
        "%s_board_main" % prefix: board_counts.get("main", 0),
        "%s_board_chinext" % prefix: board_counts.get("chinext", 0),
        "%s_board_star" % prefix: board_counts.get("star", 0),
    }, targets


monthly_rows = []
path_monthly_rows = []

for _, plan in progress_iter(update_plan_df.iterrows(), total=len(update_plan_df), desc="compare update pairs"):
    policy = plan["policy"]
    champ_end = pd.Timestamp(plan["champion_train_end"])
    chall_end = pd.Timestamp(plan["challenger_train_end"])
    if (policy, champ_end) not in model_bank or (policy, chall_end) not in model_bank:
        continue
    champ = model_bank[(policy, champ_end)]
    chall = model_bank[(policy, chall_end)]

    test_year = int(plan["test_year"])
    h1_start, h1_end = pd.Timestamp(plan["h1_start"]), pd.Timestamp(plan["h1_end"])
    h2_start, h2_end = pd.Timestamp(plan["h2_start"]), pd.Timestamp(plan["h2_end"])
    test_df = df_all[(df_all[DATE_COL] >= h1_start) & (df_all[DATE_COL] <= h2_end)].copy()
    if test_df.empty:
        continue

    champ_panel = score_panel_for_model(test_df, champ, "champion_score")
    chall_panel = score_panel_for_model(test_df, chall, "challenger_score")
    panel = champ_panel.merge(chall_panel[[STOCK_COL, DATE_COL, "challenger_score"]], on=[STOCK_COL, DATE_COL], how="left")

    prev_champ_targets = []
    prev_chall_targets = []
    for dt, gdf in panel.groupby(DATE_COL):
        dt = pd.Timestamp(dt)
        champ_res, champ_targets = calc_month_result(gdf, "champion_score", "champion", prev_champ_targets)
        chall_res, chall_targets = calc_month_result(gdf, "challenger_score", "challenger", prev_chall_targets)
        score_corr = safe_rank_ic(gdf["champion_score"], gdf["challenger_score"])
        top20_champ = gdf.sort_values("champion_score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
        top20_chall = gdf.sort_values("challenger_score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
        common = {
            "policy": policy,
            "test_year": test_year,
            "rebalance_date": dt,
            "next_date": pd.Timestamp(gdf["next_date"].iloc[0]) if "next_date" in gdf.columns and len(gdf) else pd.NaT,
            "period_half": "H1" if dt <= h1_end else "H2",
            "champion_model_id": champ["model_id"],
            "challenger_model_id": chall["model_id"],
            "champion_train_end": champ_end,
            "challenger_train_end": chall_end,
            "score_rank_corr": score_corr,
            "feature_jaccard": feature_set_jaccard(champ["feature_cols"], chall["feature_cols"]),
            "top8_overlap": len(set(champ_targets) & set(chall_targets)),
            "top8_jaccard": list_jaccard(champ_targets, chall_targets),
            "top20_overlap": len(set(top20_champ) & set(top20_chall)),
            "top20_jaccard": list_jaccard(top20_champ, top20_chall),
        }
        row = dict(common)
        row.update(champ_res)
        row.update(chall_res)
        row["update_delta_net_alpha_ret"] = row.get("challenger_net_alpha_ret", np.nan) - row.get("champion_net_alpha_ret", np.nan)
        row["challenger_better"] = bool(row["update_delta_net_alpha_ret"] > 0) if not pd.isnull(row["update_delta_net_alpha_ret"]) else False
        monthly_rows.append(row)

        # Annual champion path uses champion for both H1 and H2. Semiannual path switches to challenger in H2.
        annual_ret = row.get("champion_net_alpha_ret", np.nan)
        semi_ret = row.get("champion_net_alpha_ret", np.nan) if row["period_half"] == "H1" else row.get("challenger_net_alpha_ret", np.nan)
        path_monthly_rows.append({
            "policy": policy,
            "test_year": test_year,
            "rebalance_date": dt,
            "period_half": row["period_half"],
            "annual_champion_net_alpha_ret": annual_ret,
            "semiannual_switch_net_alpha_ret": semi_ret,
            "semi_minus_annual_ret": semi_ret - annual_ret if not pd.isnull(semi_ret) and not pd.isnull(annual_ret) else np.nan,
            "score_rank_corr": score_corr,
            "top8_overlap": row["top8_overlap"],
            "top8_jaccard": row["top8_jaccard"],
        })

        prev_champ_targets = champ_targets
        prev_chall_targets = chall_targets

monthly_df = pd.DataFrame(monthly_rows).sort_values(["policy", "test_year", "rebalance_date"])
path_monthly_df = pd.DataFrame(path_monthly_rows).sort_values(["policy", "test_year", "rebalance_date"])
monthly_df.to_csv(OUT_DIR / "v68_update_pair_monthly.csv", index=False)
path_monthly_df.to_csv(OUT_DIR / "v68_policy_path_monthly.csv", index=False)

display_df(monthly_df, 20)

## 7. 汇总：半年度 challenger 是否值得替换 champion

In [ ]:
def summarize_pair(gdf):
    h2 = gdf[gdf["period_half"] == "H2"].copy()
    out = {
        "policy": gdf["policy"].iloc[0],
        "test_year": int(gdf["test_year"].iloc[0]),
        "champion_model_id": gdf["champion_model_id"].iloc[0],
        "challenger_model_id": gdf["challenger_model_id"].iloc[0],
        "champion_train_end": gdf["champion_train_end"].iloc[0],
        "challenger_train_end": gdf["challenger_train_end"].iloc[0],
        "h2_months": int(len(h2)),
        "avg_score_rank_corr": float(h2["score_rank_corr"].mean()) if len(h2) else np.nan,
        "avg_feature_jaccard": float(h2["feature_jaccard"].mean()) if len(h2) else np.nan,
        "avg_top8_overlap": float(h2["top8_overlap"].mean()) if len(h2) else np.nan,
        "avg_top8_jaccard": float(h2["top8_jaccard"].mean()) if len(h2) else np.nan,
        "avg_top20_overlap": float(h2["top20_overlap"].mean()) if len(h2) else np.nan,
        "avg_top20_jaccard": float(h2["top20_jaccard"].mean()) if len(h2) else np.nan,
        "avg_delta_ret": float(h2["update_delta_net_alpha_ret"].mean()) if len(h2) else np.nan,
        "update_win_rate": float((h2["update_delta_net_alpha_ret"] > 0).mean()) if len(h2) else np.nan,
        "avg_champion_random_pct": float(h2["champion_random_alpha_percentile"].mean()) if len(h2) else np.nan,
        "avg_challenger_random_pct": float(h2["challenger_random_alpha_percentile"].mean()) if len(h2) else np.nan,
        "avg_champion_hit_top20": float(h2["champion_hit_top20"].mean()) if len(h2) else np.nan,
        "avg_challenger_hit_top20": float(h2["challenger_hit_top20"].mean()) if len(h2) else np.nan,
    }
    champ_sum = summarize_returns(h2["champion_net_alpha_ret"] if len(h2) else [])
    chall_sum = summarize_returns(h2["challenger_net_alpha_ret"] if len(h2) else [])
    for k, v in champ_sum.items():
        out["champion_h2_" + k] = v
    for k, v in chall_sum.items():
        out["challenger_h2_" + k] = v
    out["challenger_minus_champion_cum"] = out["challenger_h2_cum_ret"] - out["champion_h2_cum_ret"] if not pd.isnull(out["challenger_h2_cum_ret"]) and not pd.isnull(out["champion_h2_cum_ret"]) else np.nan
    out["decision_hint"] = "insufficient_h2_data"
    if out["h2_months"] >= 3:
        if out["challenger_minus_champion_cum"] > 0.02 and out["update_win_rate"] >= 0.5 and out["avg_challenger_random_pct"] >= out["avg_champion_random_pct"]:
            out["decision_hint"] = "challenger_supported"
        elif out["challenger_minus_champion_cum"] < -0.02 or out["update_win_rate"] < 0.4:
            out["decision_hint"] = "keep_champion"
        else:
            out["decision_hint"] = "neutral_observe"
    return out

pair_summary_rows = []
if len(monthly_df):
    for _, gdf in monthly_df.groupby(["policy", "test_year"]):
        pair_summary_rows.append(summarize_pair(gdf))
pair_summary_df = pd.DataFrame(pair_summary_rows).sort_values(["policy", "test_year"])
pair_summary_df.to_csv(OUT_DIR / "v68_update_pair_summary.csv", index=False)
display_df(pair_summary_df, 30)


def summarize_path(gdf):
    out = {"policy": gdf["policy"].iloc[0], "test_year": int(gdf["test_year"].iloc[0]), "months": int(len(gdf))}
    ann = summarize_returns(gdf["annual_champion_net_alpha_ret"])
    semi = summarize_returns(gdf["semiannual_switch_net_alpha_ret"])
    for k, v in ann.items():
        out["annual_" + k] = v
    for k, v in semi.items():
        out["semiannual_" + k] = v
    out["semi_minus_annual_cum"] = out["semiannual_cum_ret"] - out["annual_cum_ret"] if not pd.isnull(out["semiannual_cum_ret"]) and not pd.isnull(out["annual_cum_ret"]) else np.nan
    out["h2_update_win_rate"] = float((gdf[gdf["period_half"] == "H2"]["semi_minus_annual_ret"] > 0).mean()) if len(gdf[gdf["period_half"] == "H2"]) else np.nan
    out["avg_h2_score_rank_corr"] = float(gdf[gdf["period_half"] == "H2"]["score_rank_corr"].mean()) if len(gdf[gdf["period_half"] == "H2"]) else np.nan
    out["avg_h2_top8_overlap"] = float(gdf[gdf["period_half"] == "H2"]["top8_overlap"].mean()) if len(gdf[gdf["period_half"] == "H2"]) else np.nan
    return out

path_summary_rows = []
if len(path_monthly_df):
    for _, gdf in path_monthly_df.groupby(["policy", "test_year"]):
        path_summary_rows.append(summarize_path(gdf))
path_summary_df = pd.DataFrame(path_summary_rows).sort_values(["policy", "test_year"])
path_summary_df.to_csv(OUT_DIR / "v68_policy_path_summary.csv", index=False)
display_df(path_summary_df, 30)

## 8. 跨年份稳定性结论表

这张表用于回答：机械半年切换平均是否有正贡献，是否值得作为生产默认。

In [ ]:
stability_rows = []
for policy, gdf in pair_summary_df.groupby("policy") if len(pair_summary_df) else []:
    usable = gdf[gdf["h2_months"] >= 3].copy()
    path_usable = path_summary_df[(path_summary_df["policy"] == policy) & (path_summary_df["months"] >= 6)].copy() if len(path_summary_df) else pd.DataFrame()
    row = {
        "policy": policy,
        "usable_pairs": int(len(usable)),
        "challenger_supported_count": int((usable["decision_hint"] == "challenger_supported").sum()) if len(usable) else 0,
        "keep_champion_count": int((usable["decision_hint"] == "keep_champion").sum()) if len(usable) else 0,
        "neutral_count": int((usable["decision_hint"] == "neutral_observe").sum()) if len(usable) else 0,
        "avg_challenger_minus_champion_h2_cum": float(usable["challenger_minus_champion_cum"].mean()) if len(usable) else np.nan,
        "median_challenger_minus_champion_h2_cum": float(usable["challenger_minus_champion_cum"].median()) if len(usable) else np.nan,
        "positive_update_pair_rate": float((usable["challenger_minus_champion_cum"] > 0).mean()) if len(usable) else np.nan,
        "avg_update_win_rate": float(usable["update_win_rate"].mean()) if len(usable) else np.nan,
        "avg_score_rank_corr": float(usable["avg_score_rank_corr"].mean()) if len(usable) else np.nan,
        "avg_top8_overlap": float(usable["avg_top8_overlap"].mean()) if len(usable) else np.nan,
        "avg_feature_jaccard": float(usable["avg_feature_jaccard"].mean()) if len(usable) else np.nan,
        "avg_path_semi_minus_annual_cum": float(path_usable["semi_minus_annual_cum"].mean()) if len(path_usable) else np.nan,
        "positive_path_year_rate": float((path_usable["semi_minus_annual_cum"] > 0).mean()) if len(path_usable) else np.nan,
    }
    if row["usable_pairs"] == 0:
        row["production_hint"] = "insufficient_data"
    elif row["positive_update_pair_rate"] >= 0.6 and row["avg_challenger_minus_champion_h2_cum"] > 0:
        row["production_hint"] = "semiannual_switch_has_evidence"
    elif row["positive_update_pair_rate"] <= 0.4 or row["avg_challenger_minus_champion_h2_cum"] < 0:
        row["production_hint"] = "annual_champion_preferred"
    else:
        row["production_hint"] = "train_semiannual_but_gate_switch"
    stability_rows.append(row)

stability_summary_df = pd.DataFrame(stability_rows).sort_values("policy") if stability_rows else pd.DataFrame()
stability_summary_df.to_csv(OUT_DIR / "v68_update_stability_summary.csv", index=False)
display_df(stability_summary_df, 20)

print("saved outputs:")
for fp in sorted(OUT_DIR.glob("v68_*.csv")):
    print("-", fp)

## 9. 方法论鲁棒性矩阵：模型未来有效期

这一节不再问 challenger 是否替换 champion，而是把每个 cutoff 模型都当成同一方法论的一次抽样，观察它在未来 1-6 月、7-12 月、13-18 月是否仍有稳定 alpha。

如果方法论稳健，应该看到：

- 多数 cutoff 的未来 1-6 月为正；
- 未来 7-12 月不应系统性崩塌；
- expanding/rolling60 的结论不应完全互相矛盾；
- random percentile 和 hit top20 不应只靠少数窗口撑起来。

In [ ]:
def months_after_cutoff(rebalance_date, cutoff_date):
    d = pd.Timestamp(rebalance_date)
    c = pd.Timestamp(cutoff_date)
    return int((d.year - c.year) * 12 + (d.month - c.month))


def horizon_bucket(months_after):
    if 1 <= months_after <= 6:
        return "m01_06"
    if 7 <= months_after <= 12:
        return "m07_12"
    if 13 <= months_after <= 18:
        return "m13_18"
    return "outside"


robust_monthly_rows = []
for (policy, train_end), model_info in progress_iter(sorted(model_bank.items(), key=lambda kv: (kv[0][0], kv[0][1])), total=len(model_bank), desc="model horizon robustness"):
    cutoff = pd.Timestamp(train_end)
    test_df = df_all[(df_all[DATE_COL] > cutoff) & (df_all[DATE_COL] <= cutoff + pd.DateOffset(months=18))].copy()
    if test_df.empty:
        continue
    panel = score_panel_for_model(test_df, model_info, "score")
    prev_targets = []
    for dt, gdf in panel.groupby(DATE_COL):
        dt = pd.Timestamp(dt)
        ma = months_after_cutoff(dt, cutoff)
        hb = horizon_bucket(ma)
        if hb == "outside":
            continue
        res, targets = calc_month_result(gdf, "score", "model", prev_targets)
        robust_monthly_rows.append({
            "policy": policy,
            "model_id": model_info["model_id"],
            "train_start": model_info["train_start"],
            "train_end": cutoff,
            "train_months": model_info["train_months"],
            "feature_count": model_info["feature_count"],
            "diag_rank_ic": model_info["diag_rank_ic"],
            "rebalance_date": dt,
            "next_date": pd.Timestamp(gdf["next_date"].iloc[0]) if "next_date" in gdf.columns and len(gdf) else pd.NaT,
            "months_after_cutoff": ma,
            "horizon_bucket": hb,
            "net_alpha_ret": res.get("model_net_alpha_ret", np.nan),
            "gross_alpha_ret": res.get("model_gross_alpha_ret", np.nan),
            "trade_cost": res.get("model_trade_cost", np.nan),
            "rank_ic": res.get("model_rank_ic", np.nan),
            "random_alpha_percentile": res.get("model_random_alpha_percentile", np.nan),
            "hit_top20": res.get("model_hit_top20", np.nan),
            "target_count": res.get("model_target_count", np.nan),
            "targets": res.get("model_targets", ""),
        })
        prev_targets = targets

robust_monthly_df = pd.DataFrame(robust_monthly_rows).sort_values(["policy", "train_end", "rebalance_date"])
robust_monthly_df.to_csv(OUT_DIR / "v68_methodology_horizon_monthly.csv", index=False)

robust_summary_rows = []
if len(robust_monthly_df):
    for (policy, train_end, hb), gdf in robust_monthly_df.groupby(["policy", "train_end", "horizon_bucket"]):
        s = summarize_returns(gdf["net_alpha_ret"])
        row = {
            "policy": policy,
            "train_end": train_end,
            "horizon_bucket": hb,
            "months": int(len(gdf)),
            "avg_rank_ic": float(gdf["rank_ic"].mean()),
            "avg_random_alpha_percentile": float(gdf["random_alpha_percentile"].mean()),
            "p25_random_alpha_percentile": float(gdf["random_alpha_percentile"].quantile(0.25)),
            "avg_hit_top20": float(gdf["hit_top20"].mean()),
            "avg_feature_count": float(gdf["feature_count"].mean()),
            "avg_diag_rank_ic": float(gdf["diag_rank_ic"].mean()),
        }
        for k, v in s.items():
            row[k] = v
        robust_summary_rows.append(row)

robust_horizon_summary_df = pd.DataFrame(robust_summary_rows).sort_values(["policy", "horizon_bucket", "train_end"]) if robust_summary_rows else pd.DataFrame()
robust_horizon_summary_df.to_csv(OUT_DIR / "v68_methodology_horizon_summary.csv", index=False)

display_df(robust_horizon_summary_df, 40)

## 10. 方法论鲁棒性矩阵：相邻 cutoff 稳定性

这一节观察相邻两次训练出来的模型是否“像同一套方法论”。重点不是谁赢，而是：

- score 排名相关性是否稳定；
- top8/top20 是否过度跳变；
- 特征集合是否因为相关性去冗余而大幅变化；
- 模型差异是否对应未来表现差异。

如果相邻 cutoff 模型分歧很大且收益差异也很大，说明方法论仍然偏脆。

In [ ]:
adjacent_rows = []
for policy in sorted(set(k[0] for k in model_bank.keys())):
    ends = sorted([k[1] for k in model_bank.keys() if k[0] == policy])
    for old_end, new_end in zip(ends[:-1], ends[1:]):
        old_model = model_bank[(policy, old_end)]
        new_model = model_bank[(policy, new_end)]
        # Compare on the first 6 available rebalance months after the newer cutoff.
        test_df = df_all[(df_all[DATE_COL] > new_end) & (df_all[DATE_COL] <= new_end + pd.DateOffset(months=6))].copy()
        if test_df.empty:
            continue
        old_panel = score_panel_for_model(test_df, old_model, "old_score")
        new_panel = score_panel_for_model(test_df, new_model, "new_score")
        panel = old_panel.merge(new_panel[[STOCK_COL, DATE_COL, "new_score"]], on=[STOCK_COL, DATE_COL], how="left")
        prev_old_targets = []
        prev_new_targets = []
        for dt, gdf in panel.groupby(DATE_COL):
            dt = pd.Timestamp(dt)
            old_res, old_targets = calc_month_result(gdf, "old_score", "old", prev_old_targets)
            new_res, new_targets = calc_month_result(gdf, "new_score", "new", prev_new_targets)
            old_top20 = gdf.sort_values("old_score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
            new_top20 = gdf.sort_values("new_score", ascending=False).head(20)[STOCK_COL].astype(str).tolist()
            adjacent_rows.append({
                "policy": policy,
                "old_model_id": old_model["model_id"],
                "new_model_id": new_model["model_id"],
                "old_train_end": old_end,
                "new_train_end": new_end,
                "rebalance_date": dt,
                "score_rank_corr": safe_rank_ic(gdf["old_score"], gdf["new_score"]),
                "feature_jaccard": feature_set_jaccard(old_model["feature_cols"], new_model["feature_cols"]),
                "old_feature_count": old_model["feature_count"],
                "new_feature_count": new_model["feature_count"],
                "top8_overlap": len(set(old_targets) & set(new_targets)),
                "top8_jaccard": list_jaccard(old_targets, new_targets),
                "top20_overlap": len(set(old_top20) & set(new_top20)),
                "top20_jaccard": list_jaccard(old_top20, new_top20),
                "old_net_alpha_ret": old_res.get("old_net_alpha_ret", np.nan),
                "new_net_alpha_ret": new_res.get("new_net_alpha_ret", np.nan),
                "new_minus_old_ret": new_res.get("new_net_alpha_ret", np.nan) - old_res.get("old_net_alpha_ret", np.nan),
                "old_random_alpha_percentile": old_res.get("old_random_alpha_percentile", np.nan),
                "new_random_alpha_percentile": new_res.get("new_random_alpha_percentile", np.nan),
                "old_targets": old_res.get("old_targets", ""),
                "new_targets": new_res.get("new_targets", ""),
            })
            prev_old_targets = old_targets
            prev_new_targets = new_targets

adjacent_monthly_df = pd.DataFrame(adjacent_rows).sort_values(["policy", "new_train_end", "rebalance_date"]) if adjacent_rows else pd.DataFrame()
adjacent_monthly_df.to_csv(OUT_DIR / "v68_adjacent_cutoff_stability_monthly.csv", index=False)

adjacent_summary_rows = []
if len(adjacent_monthly_df):
    for (policy, old_end, new_end), gdf in adjacent_monthly_df.groupby(["policy", "old_train_end", "new_train_end"]):
        old_s = summarize_returns(gdf["old_net_alpha_ret"])
        new_s = summarize_returns(gdf["new_net_alpha_ret"])
        row = {
            "policy": policy,
            "old_train_end": old_end,
            "new_train_end": new_end,
            "months": int(len(gdf)),
            "avg_score_rank_corr": float(gdf["score_rank_corr"].mean()),
            "avg_feature_jaccard": float(gdf["feature_jaccard"].mean()),
            "avg_top8_overlap": float(gdf["top8_overlap"].mean()),
            "avg_top8_jaccard": float(gdf["top8_jaccard"].mean()),
            "avg_top20_overlap": float(gdf["top20_overlap"].mean()),
            "avg_top20_jaccard": float(gdf["top20_jaccard"].mean()),
            "avg_new_minus_old_ret": float(gdf["new_minus_old_ret"].mean()),
            "new_better_month_rate": float((gdf["new_minus_old_ret"] > 0).mean()),
            "avg_old_random_pct": float(gdf["old_random_alpha_percentile"].mean()),
            "avg_new_random_pct": float(gdf["new_random_alpha_percentile"].mean()),
        }
        for k, v in old_s.items():
            row["old_" + k] = v
        for k, v in new_s.items():
            row["new_" + k] = v
        row["new_minus_old_cum"] = row["new_cum_ret"] - row["old_cum_ret"] if not pd.isnull(row["new_cum_ret"]) and not pd.isnull(row["old_cum_ret"]) else np.nan
        adjacent_summary_rows.append(row)

adjacent_summary_df = pd.DataFrame(adjacent_summary_rows).sort_values(["policy", "new_train_end"]) if adjacent_summary_rows else pd.DataFrame()
adjacent_summary_df.to_csv(OUT_DIR / "v68_adjacent_cutoff_stability_summary.csv", index=False)
display_df(adjacent_summary_df, 40)

## 11. 方法论层面的最终汇总

这张表不做“是否切换模型”的上线判断，而是判断这条路径本身是否稳健：

- 未来 1-6 月是否多数 cutoff 有效；
- 未来 7-12 月是否仍有正贡献；
- 相邻 cutoff 的排序/持仓是否稳定；
- 表现是否高度依赖某一个 cutoff。

In [ ]:
methodology_rows = []
for policy in sorted(set(list(robust_horizon_summary_df["policy"]) if len(robust_horizon_summary_df) else [])):
    row = {"policy": policy}
    for hb in ["m01_06", "m07_12", "m13_18"]:
        sub = robust_horizon_summary_df[(robust_horizon_summary_df["policy"] == policy) & (robust_horizon_summary_df["horizon_bucket"] == hb)].copy()
        row["%s_cutoff_count" % hb] = int(len(sub))
        row["%s_avg_cum_ret" % hb] = float(sub["cum_ret"].mean()) if len(sub) else np.nan
        row["%s_median_cum_ret" % hb] = float(sub["cum_ret"].median()) if len(sub) else np.nan
        row["%s_positive_rate" % hb] = float((sub["cum_ret"] > 0).mean()) if len(sub) else np.nan
        row["%s_avg_random_pct" % hb] = float(sub["avg_random_alpha_percentile"].mean()) if len(sub) else np.nan
        row["%s_p25_random_pct" % hb] = float(sub["p25_random_alpha_percentile"].mean()) if len(sub) else np.nan
        row["%s_worst_cum_ret" % hb] = float(sub["cum_ret"].min()) if len(sub) else np.nan
    adj = adjacent_summary_df[adjacent_summary_df["policy"] == policy].copy() if len(adjacent_summary_df) else pd.DataFrame()
    row["adjacent_pair_count"] = int(len(adj))
    row["adjacent_avg_score_corr"] = float(adj["avg_score_rank_corr"].mean()) if len(adj) else np.nan
    row["adjacent_avg_top8_overlap"] = float(adj["avg_top8_overlap"].mean()) if len(adj) else np.nan
    row["adjacent_avg_feature_jaccard"] = float(adj["avg_feature_jaccard"].mean()) if len(adj) else np.nan
    row["adjacent_new_better_pair_rate"] = float((adj["new_minus_old_cum"] > 0).mean()) if len(adj) else np.nan

    # Conservative qualitative label for methodology robustness, not a production switch rule.
    if row.get("m01_06_cutoff_count", 0) < 3:
        row["methodology_hint"] = "insufficient_oos_windows"
    elif row.get("m01_06_positive_rate", 0) >= 0.6 and row.get("m01_06_avg_random_pct", 0) >= 0.55 and row.get("adjacent_avg_score_corr", 0) >= 0.5:
        if row.get("m07_12_positive_rate", 0) >= 0.5:
            row["methodology_hint"] = "robust_enough_for_continued_research"
        else:
            row["methodology_hint"] = "short_horizon_alpha_but_decay_risk"
    else:
        row["methodology_hint"] = "fragile_or_cutoff_sensitive"
    methodology_rows.append(row)

methodology_summary_df = pd.DataFrame(methodology_rows).sort_values("policy") if methodology_rows else pd.DataFrame()
methodology_summary_df.to_csv(OUT_DIR / "v68_methodology_robustness_summary.csv", index=False)
display_df(methodology_summary_df, 20)

print("saved expanded robustness outputs:")
for fp in sorted(OUT_DIR.glob("v68_*robustness*.csv")):
    print("-", fp)
for fp in sorted(OUT_DIR.glob("v68_adjacent_*.csv")):
    print("-", fp)
for fp in sorted(OUT_DIR.glob("v68_methodology_horizon_*.csv")):
    print("-", fp)